<a href="https://colab.research.google.com/github/Ak759-wq/NLP-project/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")

In [4]:
df = pd.DataFrame(dataset)
df.head()

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [5]:
df = df[["title", "abstract"]]

In [6]:
df = df.head(15000)

In [7]:
df["paper_text"] = df["title"] + " " + df["abstract"]

In [8]:
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

In [9]:
df["paper_text"].iloc[0]

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [10]:
df.shape

(15000, 3)

In [11]:
!pip install sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [15]:
sample_text = df["paper_text"].iloc[0]
sample_text

'Learning from compressed observations   The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regres

In [16]:
df["paper_text"]=df["paper_text"].str.replace("/n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

In [17]:
sample_embedding = model.encode(sample_text)

print(type(sample_embedding))
print(sample_embedding.shape)

<class 'numpy.ndarray'>
(384,)


In [18]:
sample_embedding[:20]

array([-0.1315641 , -0.00678266, -0.00367612,  0.03265158,  0.11219642,
        0.01227267,  0.09816719, -0.0900523 ,  0.04231161, -0.01977348,
       -0.03308417,  0.07452948,  0.10632038, -0.02060429, -0.02052106,
        0.00169493,  0.07081953,  0.05854454, -0.11231912,  0.02082474],
      dtype=float32)

In [19]:
sample_embeddings = model.encode(df["paper_text"].head(5).tolist())

In [20]:
sample_embeddings.shape

(5, 384)

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[0].reshape(1, -1)
)

print(similarity)

[[1.0000001]]


In [23]:
similarity = cosine_similarity(
    sample_embeddings[0].reshape(1, -1),
    sample_embeddings[1].reshape(1, -1)
)

print(similarity)

[[0.36625272]]


In [24]:
for i in range(1, 5):

    similarity = cosine_similarity(
        sample_embeddings[0].reshape(1, -1),
        sample_embeddings[i].reshape(1, -1)
    )

    print(f"Paper {i} Similarity:", similarity[0][0])

Paper 1 Similarity: 0.36625272
Paper 2 Similarity: 0.33522844
Paper 3 Similarity: 0.15505108
Paper 4 Similarity: 0.37421533


In [25]:
df["paper_text"] = df["title"] + " " + df["abstract"]
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.strip()

embeddings = model.encode(
    df["paper_text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [26]:
print(type(embeddings))
print(embeddings.shape)
print(embeddings.dtype)

<class 'numpy.ndarray'>
(15000, 384)
float32


In [27]:
import os
import numpy as np

In [28]:
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings...")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings...")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

Loading saved embeddings...


In [29]:
print(type(embeddings))
print(embeddings.shape)
print(embeddings.dtype)

<class 'numpy.ndarray'>
(15000, 384)
float32


In [30]:
!pip install faiss-cpu

In [31]:
import faiss

In [32]:
faiss.normalize_L2(embeddings)

In [33]:
index = faiss.IndexFlatIP(embeddings.shape[1])

In [34]:
index.add(embeddings)

In [35]:
print(index.ntotal)

15000


In [36]:
query = "Deep learning for medical image analysis"

In [37]:
query_embedding = model.encode([query])
query_embedding.shape

(1, 384)

In [38]:
faiss.normalize_L2(query_embedding)

In [39]:
D, I = index.search(query_embedding, 5)

print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [40]:
for score, idx in zip(D[0], I[0]):
    print("Title:", df.iloc[idx]["title"])
    print("Abstract:", df.iloc[idx]["abstract"][:300])
    print("Similarity Score:", round(score, 4))
    print("-" * 80)

Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image

Similarity Score: 0.6807
--------------------------------------------------------------------------------
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract:   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large 
Similarity Score: 0.6709
--------------------------------------------------------------------------------
Title: Classification of MRI dat

In [41]:
import time

start = time.time()
D, I = index.search(query_embedding, 5)
end = time.time()

print(f"Search Time: {end-start:.4f} seconds")

Search Time: 0.0052 seconds


In [42]:
def search_paper(query, k=5):

    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:300])
        print("Similarity Score:", round(score, 4))
        print("-" * 80)

    return D, I

In [43]:
search_paper("Deep learning for medical image analysis")

Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image

Similarity Score: 0.6807
--------------------------------------------------------------------------------
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract:   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for instance, a large 
Similarity Score: 0.6709
--------------------------------------------------------------------------------
Title: Classification of MRI dat

(array([[0.6807244 , 0.67092204, 0.65219975, 0.62811744, 0.61311525]],
       dtype=float32),
 array([[10466, 13730, 11873, 12691, 11282]]))

In [44]:
!pip install --upgrade transformers

In [1]:
# Explicitly uninstall and reinstall transformers and its dependencies for a clean setup.
# This ensures the correct versions are on disk and includes 'accelerate' as a common dependency.
!pip uninstall -y transformers huggingface-hub tokenizers accelerate
!pip install transformers==5.13.0 huggingface-hub==1.22.0 tokenizers==0.22.2 accelerate

from transformers import pipeline

try:
    summarizer = pipeline(
        task="summarization",
        model="facebook/bart-large-cnn",
    )
except KeyError as e:
    print(f"Caught an error: {e}")
    print("It seems the 'summarization' task is not recognized by the `transformers` library.")
    print("This usually happens when the Python environment hasn't fully reloaded the newly installed packages.")
    print("Or, a critical dependency like 'accelerate' might be missing or incorrectly installed.")
    print("\n**********************************************************************************")
    print("** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN THIS CELL. **")
    print("**********************************************************************************")
    summarizer = None # Set summarizer to None to indicate failure
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    summarizer = None

Found existing installation: transformers 5.13.0
Uninstalling transformers-5.13.0:
  Successfully uninstalled transformers-5.13.0
Found existing installation: huggingface_hub 1.22.0
Uninstalling huggingface_hub-1.22.0:
  Successfully uninstalled huggingface_hub-1.22.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached transformers-5.13.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.22.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.13.0-py3-none-any.whl (11.5 MB)
Using cached huggingface_hub-1.22.0-py3-none-any.whl (765 kB)
Using cached tokenizers-0.22.2-cp39-abi3-many

Caught an error: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"
It seems the 'summarization' task is not recognized by the `transformers` library.
This usually happens when the Python environment hasn't fully reloaded the newly installed packages.
Or, a critical dependency like 'accelerate' might be missing or incorrectly installed.

****************************************

In [2]:
type(summarizer)

NoneType

In [3]:
from transformers import pipeline

try:
    summarizer = pipeline(
        "summarization",
        model="facebook/bart-large-cnn"
    )
except KeyError as e:
    print(f"Caught an error: {e}")
    print("It seems the 'summarization' task is not recognized by the `transformers` library.")
    print("This usually happens when the Python environment hasn't fully reloaded the newly installed packages.")
    print("\n**********************************************************************************")
    print("** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN ALL CELLS. **")
    print("**********************************************************************************")
    summarizer = None # Ensure summarizer is None if initialization fails
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    summarizer = None

Caught an error: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"
It seems the 'summarization' task is not recognized by the `transformers` library.
This usually happens when the Python environment hasn't fully reloaded the newly installed packages.

**********************************************************************************
** PLEASE RESTART THE COLAB RUNTIME (Runtime

In [5]:
if 'df' not in globals():
    print("Error: DataFrame 'df' is not defined. Please rerun all cells from the beginning after a runtime restart if needed.")
    summary = None
elif summarizer is None:
    print("Error: The 'summarizer' pipeline is not initialized. This often happens when the Python environment hasn't fully reloaded the newly installed packages.\n")
    print("**********************************************************************************")
    print("** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN ALL CELLS. **")
    print("**********************************************************************************")
    summary = None
else:
    try:
        summary = summarizer(df.iloc[10466]["abstract"], max_length=120 , min_length=40)
        print(summary)
    except Exception as e:
        print(f"An error occurred during summarization: {e}")
        summary = None


Error: DataFrame 'df' is not defined. Please rerun all cells from the beginning after a runtime restart if needed.


In [6]:
type(summary)

NoneType

In [8]:
if summary is not None and len(summary) > 0:
    print(type(summary[0]))
else:
    print("The 'summary' variable is not properly initialized or is empty.")
    print("This indicates that the 'summarizer' pipeline likely failed to initialize earlier, or the 'df' DataFrame was not loaded.")
    print("\n**********************************************************************************")
    print("** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN ALL CELLS FROM THE BEGINNING. **")
    print("**********************************************************************************")


The 'summary' variable is not properly initialized or is empty.
This indicates that the 'summarizer' pipeline likely failed to initialize earlier, or the 'df' DataFrame was not loaded.

**********************************************************************************
** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN ALL CELLS FROM THE BEGINNING. **
**********************************************************************************


In [10]:
if summary is not None and len(summary) > 0:
    print(summary[0]["summary_text"])
else:
    print("The 'summary' variable is not properly initialized or is empty.")
    print("This indicates that the 'summarizer' pipeline likely failed to initialize earlier.")
    print("\n**********************************************************************************")
    print("** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN ALL CELLS FROM THE BEGINNING. **")
    print("**********************************************************************************")


The 'summary' variable is not properly initialized or is empty.
This indicates that the 'summarizer' pipeline likely failed to initialize earlier.

**********************************************************************************
** PLEASE RESTART THE COLAB RUNTIME (Runtime -> Restart runtime) AND THEN RERUN ALL CELLS FROM THE BEGINNING. **
**********************************************************************************


In [ ]:
for score,idx in zip(D[0],I[0]):
  print("Similarity score", score)
  print("Title",df.iloc[idx]["title"])
  print("Abstract",df.iloc[idx]["abstract"][:500])

  summary = summarizer(df.iloc[idx]["abstract"], max_length=120 , min_length=40)
  print(summary)
  print(summary[0]["summary_text"])
  print()

In [73]:
def search_and_summarize(query , k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score", score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()

  summary = summarizer(df.iloc[idx]["abstract"], max_length=120 , min_length=40 , do_sample=False)
  print(summary)
  print(summary[0]["summary_text"])
  print()

In [ ]:
search_and_summarize("Deep learning in medical imaging", k=5)

In [ ]:
pip install keybert==0.8.5

In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model = KeyBERT(model)

In [ ]:
type(kw_model)

In [ ]:
print(df.iloc[10466]["abstract"])

In [ ]:
text =df.iloc[10466]["abstract"]
keywords = kw_model.extract_keywords(text)

In [ ]:
print(keywords)
print(type(keywords))
print(type(keywords[0]))

In [ ]:
keywords = kw_model.extract_keywords(text , keyphrase_ngram_range=(1,3), stop_words="english")


In [72]:
print(keywords)

NameError: name 'keywords' is not defined

In [63]:
def search_and_summarize(query , k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I=index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity score", score)
    print("Title",df.iloc[idx]["title"])
    print("Abstract",df.iloc[idx]["abstract"][:500])
    print()
  summary = summarizer(df.iloc[idx]["abstract"], max_length=120 , min_length=40 , do_sample=False)
  print(summary)
  print(summary[0]["summary_text"])
  print()
  keywords = kw_model.extract_keywords(text , keyphrase_ngram_range=(1,3), stop_words="english")
  print(keywords)
  for k,s in keywords:
    print(k)
  print()

NER(Named Entity Recognition)

In [61]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 64.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [62]:
print(df.shape)

(15000, 3)


In [64]:
len(df)

15000

In [65]:
import pandas as pd
import spacy

In [66]:
nlp = spacy.load("en_core_web_sm")

In [67]:
def extract_entities(text):
    if pd.isna(text):
        return []

    text = str(text)

    doc = nlp(text)

    return [(ent.text, ent.label_) for ent in doc.ents]


In [68]:
sample_df = df.head(500).copy()

In [69]:
from datasets import load_dataset

# Check if df exists and has the 'abstract' column from the original dataset
# If not, re-initialize df and sample_df from the ML-ArXiv-Papers dataset
if 'abstract' not in df.columns or df.shape[0] != 15000:
    print("Re-initializing df with original paper data...")
    dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
    df = pd.DataFrame(dataset)
    df = df[["title", "abstract"]]
    df = df.head(15000)
    df["paper_text"] = df["title"] + " " + df["abstract"]
    df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
    df["paper_text"] = df["paper_text"].str.strip()
    sample_df = df.head(500).copy()

# Now apply the extract_entities function to the 'abstract' column
sample_df["Entities"] = sample_df["abstract"].apply(extract_entities)
print("Entities extracted successfully for sample_df!")
print(sample_df.head())

Entities extracted successfully for sample_df!
                                               title  \
0              Learning from compressed observations   
1  Sensor Networks with Random Links: Topology De...   
2  The on-line shortest path problem under partia...   
3    A neural network approach to ordinal regression   
4   Parametric Learning and Monte Carlo Optimization   

                                            abstract  \
0    The problem of statistical learning is to co...   
1    In a sensor network, in practice, the commun...   
2    The on-line shortest path problem is conside...   
3    Ordinal regression is an important type of l...   
4    This paper uncovers and explores the close r...   

                                          paper_text  \
0  Learning from compressed observations   The pr...   
1  Sensor Networks with Random Links: Topology De...   
2  The on-line shortest path problem under partia...   
3  A neural network approach to ordinal regressio...   

In [70]:
sample_df[["title", "Entities"]].head(10)

,title,Entities
0,Learning from compressed observations,"[(X$, MONEY), (X$-part, MONEY), (Y$-part, PERS..."
1,Sensor Networks with Random Links: Topology De...,"[(3, CARDINAL), (SNR, ORG), (SNR, ORG), (1, CA..."
2,The on-line shortest path problem under partia...,"[(two, CARDINAL), (1/\sqrt{n, CARDINAL)]"
3,A neural network approach to ordinal regression,"[(NNRank, ORG), (Gaussian, NORP), (NNRank, ORG..."
4,Parametric Learning and Monte Carlo Optimization,"[(Monte Carlo\nOptimization, PERSON), (MCO, OR..."
5,Preconditioned Temporal Difference Learning,"[(english, LANGUAGE), (ICML, ORG)]"
6,A Note on the Inapproximability of Correlation...,"[(two, CARDINAL), (MaxAgree, ORG), (MinDisagre..."
7,Joint universal lossy coding and identificatio...,"[(Rissanen, GPE), (Vapnik-Chervonenkis, ORG)]"
8,Supervised Feature Selection via Dependence Es...,"[(Hilbert-Schmidt Independence Criterion, ORG)..."
9,Equivalence of LP Relaxation and Max-Product f...,"[(Max, PERSON), (max, PERSON), (first, ORDINAL..."


In [71]:
sample_df.to_csv("research_papers_with_entities.csv", index=False)

print("NER completed successfully!")
print("Processed Papers:", len(sample_df))

NER completed successfully!
Processed Papers: 500


similar paper recommendation

In [58]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import pandas as pd
from datasets import load_dataset
import faiss # Ensure faiss is imported for global usage, or import locally if preferred

def similar_papers(index, top_n=5):
    global embeddings, model, df # Declare intent to use global variables

    print("--- Executing similar_papers function ---")

    # Ensure model is initialized if not already
    if 'model' not in globals() or not isinstance(model, SentenceTransformer):
        print("Initializing SentenceTransformer model inside similar_papers...")
        model = SentenceTransformer("all-MiniLM-L6-v2")

    # Ensure df is properly loaded if not already
    if 'df' not in globals() or not isinstance(df, pd.DataFrame) or 'paper_text' not in df.columns or df.shape[0] != 15000:
        print("Re-initializing df with original paper data inside similar_papers...")
        dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
        df = pd.DataFrame(dataset)
        df = df[["title", "abstract"]]
        df = df.head(15000)
        df["paper_text"] = df["title"] + " " + df["abstract"]
        df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
        df["paper_text"] = df["paper_text"].str.strip()

    # Ensure embeddings are loaded or generated
    if 'embeddings' not in globals() or not isinstance(embeddings, np.ndarray) or embeddings.shape[0] != df.shape[0]:
        print("Embeddings not found in global scope or corrupted. Attempting to load or regenerate inside similar_papers...")
        if os.path.exists("paper_embeddings.npy"):
            try:
                loaded_embeddings = np.load("paper_embeddings.npy")
                if loaded_embeddings.shape[0] == df.shape[0]:
                    embeddings = loaded_embeddings
                    print("Embeddings loaded successfully from file.")
                else:
                    raise ValueError(f"Mismatch in embedding size: expected {df.shape[0]}, got {loaded_embeddings.shape[0]}")
            except (FileNotFoundError, ValueError, Exception) as e:
                print(f"Error loading embeddings from file ({e}). Regenerating embeddings...")
                embeddings = model.encode(
                    df["paper_text"].tolist(),
                    batch_size=32,
                    show_progress_bar=True
                )
                np.save("paper_embeddings.npy", embeddings)
                print("Embeddings regenerated and saved successfully!")
        else:
            print("paper_embeddings.npy not found. Generating embeddings...")
            embeddings = model.encode(
                df["paper_text"].tolist(),
                batch_size=32,
                show_progress_bar=True
            )
            np.save("paper_embeddings.npy", embeddings)
            print("Embeddings generated and saved successfully!")

    # It's crucial that faiss is available. We'll import it here to be safe.
    # faiss.normalize_L2 modifies in-place.
    faiss.normalize_L2(embeddings)

    similarity = cosine_similarity(
        [embeddings[index]],
        embeddings
    )[0]

    similar = similarity.argsort()[::-1][1:top_n+1]

    return df.iloc[similar][["title"]]

similar_papers(0)

--- Executing similar_papers function ---


,title
208,Achievability results for statistical learning...
9237,Rate-Distortion Bounds on Bayes Risk in Superv...
2980,Sparse Signal Processing with Linear and Nonli...
10270,"Uniform Generalization, Concentration, and Ada..."
10749,On statistical learning via the lens of compre...
